# The dataset i used to train this model . 
download:  https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset

In [1]:
import pandas as pd
import spacy 

In [2]:
df_fake = pd.read_csv(r"C:\Users\ASUS\OneDrive\Desktop\Program\AI ML\dataset\news_data\Fake.csv")
df_fake.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [3]:
df_fake['label'] = 0

### Assign Labels to the Fake News Dataset

A new column named `label` is added to the fake news dataset. The value `0` is assigned to every row to indicate that the news is **fake**.

```python
df_fake['label'] = 0
```

**Explanation:**
- `df_fake['label']` creates a new column named **label**.
- `= 0` assigns the value **0** to all rows in the dataset.
- Here, **0** represents **Fake News**.

In [4]:
fake = df_fake.drop(['title','subject','date'], axis=1)
fake.head()

,text,label
0,Donald Trump just couldn t wish all Americans ...,0
1,House Intelligence Committee Chairman Devin Nu...,0
2,"On Friday, it was revealed that former Milwauk...",0
3,"On Christmas day, Donald Trump announced that ...",0
4,Pope Francis used his annual Christmas Day mes...,0


In [5]:
df_true = pd.read_csv(r"C:\Users\ASUS\OneDrive\Desktop\Program\AI ML\dataset\news_data\True.csv")
df_true.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [6]:
df_true['label'] = 1
true = df_true.drop(['title','subject','date'], axis=1)
true.head()

,text,label
0,WASHINGTON (Reuters) - The head of a conservat...,1
1,WASHINGTON (Reuters) - Transgender people will...,1
2,WASHINGTON (Reuters) - The special counsel inv...,1
3,WASHINGTON (Reuters) - Trump campaign adviser ...,1
4,SEATTLE/WASHINGTON (Reuters) - President Donal...,1


### Assign Labels to the True News Dataset

A new column named `label` is added to the True news dataset. The value `1` is assigned to every row to indicate that the news is **True**.

```python
df_true['label'] = 1
true = df_true.drop(['title','subject','date'], axis=1)
true.head()
```

**Explanation:**
- `df_true['label']` creates a new column named **label**.
- `= 1` assigns the value **1** to all rows in the dataset.
- Here, **1** represents **True News**.
- `true = df_true.drop(['title','subject','date'], axis=1)` Creating a new DataFrame and drop extra Columns. 
- `true.head()` verify that the data has been droped successfully.

In [7]:
df = pd.concat([true,fake], axis = 0,ignore_index=True)
df.head()

,text,label
0,WASHINGTON (Reuters) - The head of a conservat...,1
1,WASHINGTON (Reuters) - Transgender people will...,1
2,WASHINGTON (Reuters) - The special counsel inv...,1
3,WASHINGTON (Reuters) - Trump campaign adviser ...,1
4,SEATTLE/WASHINGTON (Reuters) - President Donal...,1


### Merge the Datasets

The `pd.concat()` function is used to combine the **true news** and **fake news** datasets into a single DataFrame.

```python
df = pd.concat([true, fake], axis=0, ignore_index=True)
df.head()
```

**Explanation:**
- `pd.concat([true, fake])` combines the two DataFrames (`true` and `fake`).
- `axis=0` merges them **row-wise** (one DataFrame below the other).
- `ignore_index=True` creates a new sequential index (0, 1, 2, ...) instead of keeping the original indices.
- `df.head()` displays the first five rows of the merged dataset to verify that the data has been combined successfully.

In [8]:
df.isnull().sum()

text     0
label    0
dtype: int64

In [9]:
df.shape

(44898, 2)

In [10]:
df = df.sample(len(df)//2 , random_state = 42)

### Used only 50% data for training 
```python
df = df.sample(len(df)//2 , random_state = 42)
```

In [11]:
df.shape

(22449, 2)

In [12]:
df.label.value_counts()

0    11784
1    10665
Name: label, dtype: int64

### Load a spacy large model

In [13]:
nlp = spacy.load("en_core_web_lg")

In [14]:
df['text_vec'] = df['text'].apply(lambda x : nlp(x).vector)

### Convert Text into Word Vectors

This step converts each news article into a numerical vector using the spaCy language model. These vectors capture the semantic meaning of the text and can be used as input features for machine learning models.

```python
df['text_vec'] = df['text'].apply(lambda x: nlp(x).vector)
```

**Explanation:**
- `df['text']` selects the **text** column from the DataFrame.
- `.apply()` applies a function to every row in the `text` column.
- `lambda x: nlp(x).vector` processes each text using the spaCy language model (`nlp`) and extracts its dense vector representation.
- `df['text_vec']` stores the generated word vectors in a new column named **text_vec**.

In [15]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(
    df.text_vec.values,
    df.label, 
    test_size=0.3, 
    random_state = 42 , 
    stratify = df.label
)

### Split the Dataset into Training and Testing Sets

The dataset is divided into **training** and **testing** sets. The training set is used to train the machine learning model, while the testing set is used to evaluate its performance on unseen data.

```python
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    df.text_vec.values,
    df.label,
    test_size=0.3,
    random_state=42,
    stratify=df.label
)
```

**Explanation:**
- `df.text_vec.values` contains the feature vectors (input data).
- `df.label` contains the target labels (`0` for Fake News and `1` for True News).
- `test_size=0.3` allocates **30%** of the data for testing and **70%** for training.
- `random_state=42` ensures that the data is split in the same way every time the code is executed, making the results reproducible.
- `stratify=df.label` preserves the class distribution in both the training and testing sets, ensuring that the proportion of fake and true news remains the same in each set.

In [16]:
x_train.shape , y_train.shape

((15714,), (15714,))

In [17]:
x_test.shape , y_test.shape

((6735,), (6735,))

In [18]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [19]:
x_train_2d = np.stack(x_train)
x_test_2d = np.stack(x_test)

### Convert Feature Vectors into 2D NumPy Arrays

The text vectors are converted into two-dimensional NumPy arrays so they can be used as input for machine learning models.

```python
x_train_2d = np.stack(x_train)
x_test_2d = np.stack(x_test)
```

**Explanation:**
- `x_train` and `x_test` contain the text vectors generated by the spaCy model.
- `np.stack()` combines the individual vectors into a single **2D NumPy array**.
- `x_train_2d` stores the training feature vectors.
- `x_test_2d` stores the testing feature vectors.
- Each row in the resulting array represents one news article, and each column represents one feature of its vector embedding.

In [20]:
x_train_2d.shape , x_test_2d.shape

((15714, 300), (6735, 300))

In [21]:
x_test_2d

array([[-2.2954662 ,  0.5777568 , -1.8070658 , ..., -2.7833896 ,
        -1.6871544 ,  1.6493589 ],
       [-1.7683676 ,  0.52526724, -1.7620945 , ..., -1.0667852 ,
        -2.7680383 ,  0.8842659 ],
       [-1.2841464 , -0.96470594,  0.01072613, ..., -1.2438539 ,
        -1.8610109 ,  0.92251056],
       ...,
       [-2.3367145 ,  0.87643087, -2.2679245 , ..., -1.6276816 ,
        -1.9281343 ,  0.98777264],
       [-1.5124607 ,  0.7738136 , -1.5104547 , ..., -0.8290069 ,
        -2.2638047 ,  0.65563667],
       [-3.4663296 , -0.19155852, -1.2418528 , ..., -3.1140804 ,
        -0.83600086, -0.18455116]], dtype=float32)

In [22]:
scale = MinMaxScaler()
scale_x_train = scale.fit_transform(x_train_2d)
scale_x_test = scale.transform(x_test_2d)

### Feature Scaling

The feature vectors are normalized using the **MinMaxScaler** to scale all values between **0 and 1**, improving the performance of machine learning models.

```python
from sklearn.preprocessing import MinMaxScaler

scale = MinMaxScaler()

scale_x_train = scale.fit_transform(x_train_2d)
scale_x_test = scale.transform(x_test_2d)
```

**Explanation:**
- `fit_transform()` scales the training data.
- `transform()` scales the testing data using the same parameters.

# model Training

In [23]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report

In [24]:
clf = MultinomialNB()
clf.fit(scale_x_train,y_train)

MultinomialNB()

### Train the Multinomial Naive Bayes Model

The **Multinomial Naive Bayes** classifier is trained using the scaled training data.

```python
from sklearn.naive_bayes import MultinomialNB

clf = MultinomialNB()
clf.fit(scale_x_train, y_train)
```

**Explanation:**
- `MultinomialNB()` creates the Naive Bayes model.
- `fit()` trains the model using the training features and labels.

In [25]:
y_pred = clf.predict(scale_x_test)
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.89      0.83      0.86      3535
           1       0.83      0.89      0.86      3200

    accuracy                           0.86      6735
   macro avg       0.86      0.86      0.86      6735
weighted avg       0.86      0.86      0.86      6735



### Model Evaluation

The trained model is evaluated on the test dataset using the **classification report**, which provides performance metrics such as precision, recall, F1-score, and accuracy.

```python
from sklearn.metrics import classification_report

y_pred = clf.predict(scale_x_test)
print(classification_report(y_test, y_pred))
```

**Results:**

- **Accuracy:** 86%
- **Class 0 (Fake News):**
  - Precision: **0.89**
  - Recall: **0.83**
  - F1-score: **0.86**
- **Class 1 (True News):**
  - Precision: **0.83**
  - Recall: **0.89**
  - F1-score: **0.86**

In [26]:
clf = KNeighborsClassifier(
    n_neighbors=4,
    weights='distance',
    metric='euclidean',
    )

clf.fit(scale_x_train,y_train)

KNeighborsClassifier(metric='euclidean', n_neighbors=4, weights='distance')

### Train the K-Nearest Neighbors (KNN) Model

The **K-Nearest Neighbors (KNN)** classifier is trained using the scaled training data.

```python
from sklearn.neighbors import KNeighborsClassifier

clf = KNeighborsClassifier(
    n_neighbors=4,
    weights='distance',
    metric='euclidean'
)

clf.fit(scale_x_train, y_train)
```

**Explanation:**
- `n_neighbors=4` uses the 4 nearest neighbors for classification.
- `weights='distance'` gives more importance to closer neighbors.
- `metric='euclidean'` uses Euclidean distance to measure similarity.
- `fit()` trains the KNN model on the training data.

In [27]:
y_predicted = clf.predict(scale_x_test)
print(classification_report(y_test, y_predicted))


              precision    recall  f1-score   support

           0       0.97      0.97      0.97      3535
           1       0.96      0.97      0.96      3200

    accuracy                           0.97      6735
   macro avg       0.97      0.97      0.97      6735
weighted avg       0.97      0.97      0.97      6735



### Evaluate the KNN Model

The trained KNN model is evaluated on the test dataset using the classification report.

```python
from sklearn.metrics import classification_report

y_predicted = clf.predict(scale_x_test)
print(classification_report(y_test, y_predicted))
```

**Results:**
- **Accuracy:** **97%**
- **Precision:** 97% (Fake News), 96% (True News)
- **Recall:** 97% (Fake News), 97% (True News)
- **F1-Score:** 97% (Fake News), 96% (True News)

The KNN model achieved an overall **97% accuracy**, demonstrating excellent performance in classifying fake and true news articles.